# Problem Statement

Fine-tune a BERT model to classify news headlines into topic categories using the AG News dataset.

# install

In [2]:
!pip install transformers datasets torch scikit-learn

  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
Using cached transformers-5.6.2-py3-none-any.whl (10.4 MB)


# imports

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Load Dataset

In [4]:
dataset = load_dataset("ag_news")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

# Tokenization

In [5]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example['text'], padding='max_length', truncation=True)

tokenized_dataset = dataset.map(tokenize, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

# Reduce Dataset

In [6]:
small_train = tokenized_dataset['train'].shuffle(seed=42).select(range(2000))
small_test = tokenized_dataset['test'].shuffle(seed=42).select(range(500))

# Model

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Metrics

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average='weighted')
    }

# Training Arguments

In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
)

# Trainer

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics
)

# Train

In [11]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.5635924682617187, metrics={'train_runtime': 109.8495, 'train_samples_per_second': 18.207, 'train_steps_per_second': 2.276, 'total_flos': 264944246784000.0, 'train_loss': 0.5635924682617187, 'epoch': 1.0})

# Evaluate

In [12]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy,F1
No log,0.423176,250,0.862000,0.861686


{'eval_loss': 0.42317625880241394,
 'eval_accuracy': 0.862,
 'eval_f1': 0.8616859008294512}

# Save Model

In [13]:
trainer.save_model("bert-news")
tokenizer.save_pretrained("bert-news")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert-news/tokenizer_config.json', 'bert-news/tokenizer.json')

# Test Prediction

In [15]:
text = "Apple launches new AI product"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

# Move inputs to the same device as the model
inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model(**inputs)

pred = outputs.logits.argmax().item()

labels = ["World", "Sports", "Business", "Sci/Tech"]
print("Prediction:", labels[pred])

Prediction: Sci/Tech


In [16]:
import shutil

shutil.make_archive("bert-news", 'zip', "bert-news")

'/content/bert-news.zip'

In [17]:
from google.colab import files
files.download("bert-news.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>